# ARMT with Llama-3.2-1B!

## Create a model from config in ARMT official repo

In [1]:
# Minimal example: wrap Llama-3.2-1B with ARMT and run a quick forward pass
import os
import torch
from transformers import AutoTokenizer
from modeling_amt.model import ARMTConfig, ARMTForCausalLM

# Base model to wrap
base_model_name = "meta-llama/Llama-3.2-1B"

# Minimal ARMT config, similar to run_finetuning_lm_rmt_hf_armt.py
segment_size = 128
armt_cfg = ARMTConfig(
    base_model_name=base_model_name,
    num_mem_tokens=16,
    d_mem=64,
    segment_size=segment_size,
    segment_alignment="left",
    sliding_window=True,
    attend_to_previous_input=False,
    use_sink=True,
    layers_attr="model.layers",  # Llama layers path
    wrap_pos=False,
    correction=True,
    n_heads=1,
    use_denom=True,
    gating=False,
    freeze_mem=False,
)

# Build wrapped model
model = ARMTForCausalLM(armt_cfg)
model.eval()

# Tokenizer and a tiny test batch
# Llama tokenizer may not have pad_token by default, set it to eos for batching
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer([
    "Hello Llama with ARMT!",
    "Testing ARMT wrapping on Llama-3.2-1B.",
], return_tensors="pt", padding=True, truncation=True, max_length=segment_size)

labels = inputs["input_ids"].clone()
with torch.no_grad():
    out = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        labels=labels,
    )
print("loss:", float(out.loss))


*** Setting default RWKV_MY_TESTING = x060 ***
[2025-08-11 17:01:38,641] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
nvcc warning : incompatible redefinition for option 'compiler-bindir', the last value of this option was used
/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/ivan.rodkin/miniconda3/envs/env/lib/libcufile.so: undefined reference to `dlvsym'
/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/ivan.rodkin/miniconda3/envs/env/lib/libcufile.so: undefined reference to `dlopen'
/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/ivan.rodkin/miniconda3/envs/env/lib/libcufile.so: undefined reference to `dlclose'
/home/ivan.rodkin/miniconda3/envs

[RWKV.model] Running RWKV infctx using 'torch-jit' with torch '2.3.1+cu121'
[RWKV.model] Running RWKV infctx using 'torch-jit' with torch '2.3.1+cu121'
*** Can't import RWKV model ***
loss: 7.721312999725342


## Importnat: now model needs to be trained. The memory weights are initialized from scratch.

## Pushing to hub with the corresponding code

In [2]:
# Push the wrapped ARMT+Llama model to Hugging Face Hub
import os
import json
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory

from huggingface_hub import HfApi, upload_folder
from transformers import AutoModelForCausalLM

# Change to your namespace/repo name if desired
repo_id = "irodkin/armt-sw-llama-3.2-1b-untrained"

# Local source for ARMT code in this repo (relative to this notebook directory)
local_modeling_dir = Path("./modeling_amt")
assert (local_modeling_dir / "model.py").exists(), "Expected modeling_amt/model.py next to this notebook"

with TemporaryDirectory() as tmpdir:
    repo_dir = Path(tmpdir) / "repo"
    repo_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights and config
    model.save_pretrained(repo_dir)

    # Ensure auto_map uses a single-file entry point
    cfg_path = repo_dir / "config.json"
    cfg = json.loads(cfg_path.read_text())
    cfg["architectures"] = ["ARMTForCausalLM"]
    cfg["auto_map"] = {
        "AutoConfig": "armt_entry.ARMTConfig",
        "AutoModelForCausalLM": "armt_entry.ARMTForCausalLM",
    }
    cfg_path.write_text(json.dumps(cfg, indent=2))

    # Include ARMT implementation package
    remote_pkg_dir = repo_dir / "modeling_amt"
    remote_pkg_dir.mkdir(parents=True, exist_ok=True)
    (remote_pkg_dir / "__init__.py").write_text("")
    shutil.copy2(local_modeling_dir / "model.py", remote_pkg_dir / "model.py")
    shutil.copy2(local_modeling_dir / "language_modeling.py", remote_pkg_dir / "language_modeling.py")

    # Single-file entry for auto_map
    (repo_dir / "armt_entry.py").write_text(
        "from modeling_amt.model import ARMTForCausalLM, ARMTConfig\n"
    )

    # README
    (repo_dir / "README.md").write_text(
        f"# {repo_id}\n\nARMT-wrapped Llama-3.2-1B demo. Load with trust_remote_code=True. WARNING: This model is initialized with random weights."
    )

    # Create or update repo and upload
    api = HfApi()
    api.create_repo(repo_id, private=True, exist_ok=True)
    upload_folder(
        folder_path=str(repo_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload ARMT+Llama-3.2-1B demo",
    )

print(f"Pushed to {repo_id}")



/home/ivan.rodkin/miniconda3/envs/env/lib/python3.9/site-packages/huggingface_hub/hf_api.py:9696: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Pushed to irodkin/armt-sw-llama-3.2-1b-untrained


A new version of the following files was downloaded from https://huggingface.co/irodkin/armt-sw-llama-3.2-1b-untrained:
- armt_entry.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/243M [00:00<?, ?B/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of ARMTForCausalLM were not initialized from the model checkpoint at irodkin/armt-sw-llama-3.2-1b-untrained and are newly initialized: ['armt.memory_cell.model.lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<class 'modeling_amt.model.ARMTForCausalLM'>


## Now you can load the model anywhere

In [8]:
from transformers import AutoModelForCausalLM

loaded_model = AutoModelForCausalLM.from_pretrained(
    "irodkin/armt-sw-llama-3.2-1b-untrained", trust_remote_code=True
)

loaded_model.tie_weights()

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of ARMTForCausalLM were not initialized from the model checkpoint at irodkin/armt-sw-llama-3.2-1b-untrained and are newly initialized: ['armt.memory_cell.model.lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
